## Sample Code to vectorize using tf-idf and one-hot-encoding

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Sample text documents
documents = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
]

# Create TfidfVectorizer object
vectorizer = TfidfVectorizer()

# Fit and transform the documents
tfidf_matrix = vectorizer.fit_transform(documents)

# Get feature names (words)
feature_names = vectorizer.get_feature_names_out()

# Print the TF-IDF matrix
tfidf_list = tfidf_matrix.toarray()
print(tfidf_matrix.toarray())
print(feature_names)

[[0.         0.46979139 0.58028582 0.38408524 0.         0.
  0.38408524 0.         0.38408524]
 [0.         0.6876236  0.         0.28108867 0.         0.53864762
  0.28108867 0.         0.28108867]
 [0.51184851 0.         0.         0.26710379 0.51184851 0.
  0.26710379 0.51184851 0.26710379]
 [0.         0.46979139 0.58028582 0.38408524 0.         0.
  0.38408524 0.         0.38408524]]
['and' 'document' 'first' 'is' 'one' 'second' 'the' 'third' 'this']


In [2]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import CountVectorizer

# Sample sentences
documents = [
    "This is the first document.",
    "This document is the second document.",
    "And this is the third one.",
    "Is this the first document?",
]

# Create a vocabulary from the sentences
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(documents)

# Initialize and fit the OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
one_hot_encoded = encoder.fit_transform(X.toarray())

# Get feature names
feature_names = vectorizer.get_feature_names_out()

print("Shape of one-hot encoded matrix:", one_hot_encoded.shape)
print(one_hot_encoded)
print("Feature names (words):", feature_names)

Shape of one-hot encoded matrix: (4, 16)
[[1. 0. 0. 1. 0. 0. 1. 1. 1. 0. 1. 0. 1. 1. 0. 1.]
 [1. 0. 0. 0. 1. 1. 0. 1. 1. 0. 0. 1. 1. 1. 0. 1.]
 [0. 1. 1. 0. 0. 1. 0. 1. 0. 1. 1. 0. 1. 0. 1. 1.]
 [1. 0. 0. 1. 0. 0. 1. 1. 1. 0. 1. 0. 1. 1. 0. 1.]]
Feature names (words): ['and' 'document' 'first' 'is' 'one' 'second' 'the' 'third' 'this']


In [3]:
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
print(cosine_similarity(tfidf_list[0], tfidf_list[1]))
print(cosine_similarity(one_hot_encoded[0], one_hot_encoded[1]))

0.6469256761082494
0.6666666666666666


## Assignment 2: text encoding and semantic similarity

I use the sentence file. Clauses are much shorter after stopword removal (median 3 content words vs 6 for sentences), so there is less to match on.

In [4]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
from gensim.models import Word2Vec

SEED = 42
DATA_FILE = "emotion_multigenre_corpus_setences.txt"
STOPLIST = "mallet_en_stoplist.txt"
GENRES = ["newsheadline", "moviereview", "blog"]
TOP_K = 10
N_QUERIES_PER_GENRE = 3
N_SAMPLE_PER_GENRE = 100

pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

### Load the corpus
Tab-separated. The last three fields are text, emotion and genre (one line has an extra source column, so I read from the end).

In [5]:
def load_corpus(path):
    # read the corpus into a dataframe
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 4:
                continue
            rows.append({"id": parts[0], "text": parts[-3].strip(), "emotion": parts[-2], "genre": parts[-1]})
    return pd.DataFrame(rows)

df = load_corpus(DATA_FILE)
print("Raw rows:", len(df))
print(df.genre.value_counts(), "\n")
print(df.emotion.value_counts())

Raw rows: 17220
genre
moviereview     11855
blog             4115
newsheadline     1250
Name: count, dtype: int64 

emotion
joy             5670
disgust         5455
noemotion       3967
sadness          653
surprise         436
anticipation     435
fear             263
anger            240
trust             75
other             26
Name: count, dtype: int64


### Preprocessing
Lowercase, keep alphabetic tokens, remove the Mallet stopwords and single letters. I also drop contractions (movie reviews write `did n't`, blogs write `didn't`) and the `-LRB-`/`-RRB-` bracket codes in the movie reviews. Empty and duplicate sentences are removed.

In [6]:
with open(STOPLIST, encoding="utf-8") as f:
    STOPWORDS = set(w.strip() for w in f if w.strip())

TOKEN_RE = re.compile(r"[a-z]+(?:'[a-z]+)?")
PTB_BRACKETS = re.compile(r"-[LR][RSC]B-")

def tokenize(text):
    # lowercase content words only
    toks = TOKEN_RE.findall(PTB_BRACKETS.sub(" ", text).lower())
    return [t for t in toks if t not in STOPWORDS and len(t) > 1 and "'" not in t]

df["tokens"] = df.text.apply(tokenize)
before = len(df)
df = df[df.tokens.str.len() > 0]
df = df.drop_duplicates(subset="text").reset_index(drop=True)
print(f"Kept {len(df)} of {before} sentences")
print(df.groupby("genre").tokens.apply(lambda s: s.str.len().mean()).rename("avg content tokens"))
df[["text", "tokens", "genre"]].head()

Kept 16822 of 17220 sentences
genre
blog            5.015490
moviereview     8.098854
newsheadline    4.659609
Name: avg content tokens, dtype: float64


,text,tokens,genre
0,The Rock is destined to be the 21st Century 's new `` Conan '' and that he 's going to make a splash even ...,"[rock, destined, st, century, conan, make, splash, greater, arnold, schwarzenegger, jean, claud, van, damm...",moviereview
1,The gorgeously elaborate continuation of `` The Lord of the Rings '' trilogy is so huge that a column of w...,"[gorgeously, elaborate, continuation, lord, rings, trilogy, huge, column, words, adequately, describe, wri...",moviereview
2,Effective but too-tepid biopic,"[effective, tepid, biopic]",moviereview
3,"If you sometimes like to go to the movies to have fun , Wasabi is a good place to start .","[movies, fun, wasabi, good, place, start]",moviereview
4,"Emerges as something rare , an issue movie that 's so honest and keenly observed that it does n't feel lik...","[emerges, rare, issue, movie, honest, keenly, observed, feel]",moviereview


### Encoders
- One-hot: `CountVectorizer(binary=True)`. The `OneHotEncoder` in the sample above encodes the count values of each column, not whether a word is present, and it gets very large on the full corpus.
- TF-IDF: `TfidfVectorizer`.
- Word2Vec: skip-gram, trained on the sentences being compared. Sentence vector = mean of its word vectors.

In [7]:
identity = lambda x: x

def encode_onehot(token_lists):
    # binary bag of words
    vec = CountVectorizer(analyzer=identity, binary=True)
    return vec.fit_transform(token_lists), vec

def encode_tfidf(token_lists):
    # tf-idf weighted bag of words
    vec = TfidfVectorizer(analyzer=identity)
    return vec.fit_transform(token_lists), vec

def encode_w2v(token_lists, vector_size=100, window=5, min_count=1, epochs=None):
    # train Word2Vec and average word vectors per sentence
    if epochs is None:
        epochs = 50 if len(token_lists) < 1000 else 20
    model = Word2Vec(sentences=token_lists, vector_size=vector_size, window=window,
                     min_count=min_count, sg=1, epochs=epochs, workers=1, seed=SEED)
    X = np.zeros((len(token_lists), vector_size))
    for i, toks in enumerate(token_lists):
        vecs = [model.wv[t] for t in toks if t in model.wv]
        if vecs:
            X[i] = np.mean(vecs, axis=0)
    return X, model

ENCODERS = {"one-hot": encode_onehot, "tf-idf": encode_tfidf, "word2vec": encode_w2v}

### Search helpers

In [8]:
def top_k_similar(X, q_idx, frame, k=TOP_K):
    # top k rows by cosine similarity to row q_idx (query itself excluded)
    sims = cos_sim(X[q_idx:q_idx + 1], X).ravel()
    sims[q_idx] = -np.inf
    order = np.argsort(-sims, kind="stable")[:k]
    out = frame.iloc[order][["text", "genre", "emotion"]].copy()
    out.insert(0, "score", sims[order].round(4))
    out.insert(0, "rank", range(1, len(order) + 1))
    return out.reset_index().rename(columns={"index": "row"})

def pick_queries(frame, n=N_QUERIES_PER_GENRE, min_tokens=3, seed=SEED):
    # n random queries per genre with at least min_tokens content words
    genres = [g for g in GENRES if g in set(frame.genre)]
    picks = []
    for g in genres:
        pool = frame[(frame.genre == g) & (frame.tokens.str.len() >= min_tokens)]
        picks += pool.sample(n, random_state=seed).index.tolist()
    return picks

def run_retrieval(frame, matrices, query_idx, label, show=True):
    # print the top 10 for every query and method, return simple stats
    records = []
    for q in query_idx:
        qrow = frame.loc[q]
        if show:
            print("=" * 110)
            print(f"[{label}] QUERY ({qrow.genre}, {qrow.emotion}): {qrow.text}")
        for name, X in matrices.items():
            res = top_k_similar(X, frame.index.get_loc(q), frame)
            if show:
                print(f"\n--- {name} ---")
                print(res[["rank", "score", "genre", "emotion", "text"]].to_string(index=False))
            records.append({
                "setting": label, "query_row": q, "query_genre": qrow.genre, "method": name,
                "same_genre@10": (res.genre == qrow.genre).mean(),
                "same_emotion@10": (res.emotion == qrow.emotion).mean(),
                "mean_score@10": res.score.mean(),
            })
        if show:
            print()
    return pd.DataFrame(records)

## Experiment 1: entire corpus

In [9]:
full_mats = {}
for name, enc in ENCODERS.items():
    X, _ = enc(df.tokens.tolist())
    full_mats[name] = X
    print(f"{name:9s} matrix shape: {X.shape}")

queries_full = pick_queries(df)
df.loc[queries_full, ["genre", "emotion", "text"]]

one-hot   matrix shape: (16822, 20787)
tf-idf    matrix shape: (16822, 20787)


word2vec  matrix shape: (16822, 100)


,genre,emotion,text
12004,newsheadline,noemotion,Depp to star in spy poisoning film
11846,newsheadline,anger,Vaccine mandate upsets legislators
12485,newsheadline,joy,Murphys rule box office with 'Norbit'
3477,moviereview,joy,"You may leave the theater with more questions than answers , but darned if your toes wo n't still be tappi..."
1799,moviereview,joy,"Despite its shortcomings , Girls Ca n't Swim represents an engaging and intimate first feature by a talent..."
8684,moviereview,disgust,"While Benigni -LRB- who stars and co-wrote -RRB- seems to be having a wonderful time , he might be alone i..."
13020,blog,joy,"I was planning on going to Greece sometime anyway, so we spent three days in Athens, and two and a half in..."
15267,blog,noemotion,There are moments in our lives when we find ourselves at a crossroad.
14740,blog,noemotion,I invited them in and ended up with lots of pamphlets the book of Mormon.


In [10]:
res_full = run_retrieval(df, full_mats, queries_full, label="entire corpus")

[entire corpus] QUERY (newsheadline, noemotion): Depp to star in spy poisoning film

--- one-hot ---
 rank  score        genre   emotion                                                                                                      text
    1 0.5477 newsheadline noemotion                                                                Johnny Depp to make movie of spy poisoning
    2 0.5071 newsheadline noemotion                                                       Variety: Johnny Depp to make movie of spy poisoning
    3 0.4472  moviereview   disgust                                                                                    Do not see this film .
    4 0.4472  moviereview noemotion                                                                                                  Spy-vs .
    5 0.3651 newsheadline noemotion                                                            A film star in Kampala, conjuring aminos ghost
    6 0.3381  moviereview   disgust            


--- one-hot ---
 rank  score genre      emotion                                                                                                         text
    1 0.2673  blog    noemotion Lance and I spent the better part of the next three days lounging around Zurich and drinking in Switzerland.
    2 0.2673  blog anticipation                                                I only know that it could be anytime within the next 60 days.
    3 0.2673  blog anticipation                                              I will get that 5 days and then later on he will still get RnR.
    4 0.2673  blog    noemotion                                                                       Well, we're half-way through our trip.
    5 0.2673  blog    noemotion                                                               So, that's what I did for like a half an hour.
    6 0.2673  blog    noemotion                                                              But two days back to back was more than enou


--- word2vec ---
 rank  score        genre   emotion                                                                                                                                            text
    1 0.9133  moviereview noemotion                                                                                  New Yorkers always seem to find the oddest places to dwell ...
    2 0.8963  moviereview   disgust                                                    Those unfamiliar with Mormon traditions may find The Singles Ward occasionally bewildering .
    3 0.8959         blog noemotion                                                                                                         I dunno, press the button and find out.
    4 0.8952         blog      fear                                                                                      Not to mention I find her to be incredibly intimidatingsp.
    5 0.8923 newsheadline noemotion                                               

Share of the top 10 from the query's genre, next to each genre's share of the corpus:

In [11]:
base = df.genre.value_counts(normalize=True).rename("base rate (random retrieval)")
print(base.round(3), "\n")
summary_full = (res_full.groupby(["query_genre", "method"])
                [["same_genre@10", "same_emotion@10", "mean_score@10"]].mean().round(3))
summary_full

genre
moviereview     0.701
blog            0.226
newsheadline    0.073
Name: base rate (random retrieval), dtype: float64 



same_genre@10  same_emotion@10  mean_score@10
query_genre  method                                                 
blog         one-hot           0.700            0.400          0.291
             tf-idf            0.600            0.267          0.235
             word2vec          0.733            0.300          0.923
moviereview  one-hot           0.633            0.300          0.358
             tf-idf            0.467            0.267          0.309
             word2vec          0.800            0.467          0.937
newsheadline one-hot           0.533            0.300          0.302
             tf-idf            0.533            0.333          0.290
             word2vec          0.400            0.167          0.946

Top-10 overlap between methods (Jaccard, averaged over the 9 queries):

In [12]:
def method_overlap(frame, matrices, query_idx):
    # mean Jaccard overlap of top-10 sets for each pair of methods
    pairs = [("one-hot", "tf-idf"), ("one-hot", "word2vec"), ("tf-idf", "word2vec")]
    rows = []
    for q in query_idx:
        qi = frame.index.get_loc(q)
        sets = {m: set(top_k_similar(X, qi, frame).row) for m, X in matrices.items()}
        for a, b in pairs:
            rows.append({"pair": f"{a} vs {b}", "jaccard": len(sets[a] & sets[b]) / len(sets[a] | sets[b])})
    return pd.DataFrame(rows).groupby("pair").jaccard.mean().round(3)

method_overlap(df, full_mats, queries_full)

pair
one-hot vs tf-idf      0.543
one-hot vs word2vec    0.050
tf-idf vs word2vec     0.030
Name: jaccard, dtype: float64

Nearest words in the full-corpus Word2Vec model:

In [13]:
w2v_full = encode_w2v(df.tokens.tolist())[1]
for w in ["film", "funny", "sad", "war", "love", "boring"]:
    if w in w2v_full.wv:
        print(f"{w:8s} ->", [n for n, _ in w2v_full.wv.most_similar(w, topn=6)])

film     -> ['movie', 'result', 'features', 'attractive', 'fairly', 'filmmaking']
funny    -> ['thinks', 'amusing', 'goofy', 'consistently', 'cute', 'darkly']
sad      -> ['sack', 'knockaround', 'count', 'sour', 'sick', 'rude']
war      -> ['ii', 'current', 'criminal', 'civil', 'china', 'climate']
love     -> ['drunk', 'side', 'praise', 'triangle', 'americans', 'punch']
boring   -> ['joke', 'rooting', 'lifeless', 'plain', 'depressing', 'frustrating']


## Experiment 2: 100 sentences per genre
Each genre gets its own matrices and Word2Vec model (trained on its 100 sentences only). Each query is compared with the other 99 sentences of its genre.

In [14]:
sample = df.groupby("genre").sample(N_SAMPLE_PER_GENRE, random_state=SEED)

res_small = []
for g in GENRES:
    sub = sample[sample.genre == g]
    mats = {name: enc(sub.tokens.tolist())[0] for name, enc in ENCODERS.items()}
    print(f"\n##### GENRE: {g}  (vocab size: {mats['one-hot'].shape[1]}) #####")
    q_idx = pick_queries(sub)
    res_small.append(run_retrieval(sub, mats, q_idx, label=f"100-{g}"))
res_small = pd.concat(res_small, ignore_index=True)


##### GENRE: newsheadline  (vocab size: 417) #####
[100-newsheadline] QUERY (newsheadline, noemotion): Napping is good for your heart, study finds

--- one-hot ---
 rank  score        genre   emotion                                                        text
    1 0.3162 newsheadline      fear Women Face Greatest Threat of Violence at Home, Study Finds
    2 0.2236 newsheadline noemotion                          Cholesterol link in diabetes study
    3 0.2000 newsheadline noemotion                  Stanford study warns of Internet addiction
    4 0.1826 newsheadline noemotion        Experts issue new heart disease guidelines for women
    5 0.1690 newsheadline noemotion    Study: men's sweat triggers high sexual arousal in women
    6 0.0000 newsheadline noemotion                                   Ice storm coating Midwest
    7 0.0000 newsheadline noemotion                    Male sweat boosts women's hormone levels
    8 0.0000 newsheadline       joy                      Making pea


##### GENRE: moviereview  (vocab size: 704) #####
[100-moviereview] QUERY (moviereview, disgust): One of those films that seems tailor made to air on pay cable to offer some modest amusements when one has nothing else to watch .

--- one-hot ---
 rank  score       genre   emotion                                                                                                                                                                                                                                 text
    1 0.0877 moviereview   disgust                                                                                                      A late-night cable sexploitation romp masquerading as a thriller about the ruthless social order that governs college cliques .
    2 0.0745 moviereview   disgust                       Like many Western action films , this thriller is too loud and thoroughly overbearing , but its heartfelt concern about North Korea 's recent past and South Korea 's fu


##### GENRE: blog  (vocab size: 486) #####
[100-blog] QUERY (blog, noemotion): I struggle with the Catholic concept of baptism being a gift we give infants because it is God's gift to us.

--- one-hot ---
 rank  score genre      emotion                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           text
    1  0.126  blog         fear                                                                                                                                                                                                                                            

In [15]:
summary_small = (res_small.groupby(["query_genre", "method"])
                 [["same_emotion@10", "mean_score@10"]].mean().round(3))
summary_small

same_emotion@10  mean_score@10
query_genre  method                                  
blog         one-hot             0.133          0.030
             tf-idf              0.133          0.026
             word2vec            0.167          0.990
moviereview  one-hot             0.433          0.046
             tf-idf              0.433          0.036
             word2vec            0.367          0.998
newsheadline one-hot             0.567          0.050
             tf-idf              0.567          0.041
             word2vec            0.700          0.902

Every result above comes from the query's genre, so for the genre question I also encode the 300 sampled sentences together (same 9 queries). Random retrieval would give 1/3.

In [16]:
pooled = sample.copy()
pooled_mats = {name: enc(pooled.tokens.tolist())[0] for name, enc in ENCODERS.items()}
queries_pooled = pick_queries(pooled)
res_pooled = run_retrieval(pooled, pooled_mats, queries_pooled, label="pooled 300", show=False)
(res_pooled.groupby(["query_genre", "method"])
 [["same_genre@10", "same_emotion@10", "mean_score@10"]].mean().round(3))

same_genre@10  same_emotion@10  mean_score@10
query_genre  method                                                 
blog         one-hot           0.700            0.100          0.065
             tf-idf            0.700            0.100          0.050
             word2vec          0.267            0.100          0.997
moviereview  one-hot           0.333            0.333          0.089
             tf-idf            0.333            0.333          0.073
             word2vec          0.700            0.433          0.998
newsheadline one-hot           0.200            0.400          0.099
             tf-idf            0.200            0.400          0.084
             word2vec          0.267            0.467          0.996

## Summary
Word2Vec scores are always high because averaged vectors all point in a similar direction, so they can only be compared within the method.

In [17]:
compare = pd.concat({
    "entire corpus": res_full.groupby("method")[["same_genre@10", "same_emotion@10", "mean_score@10"]].mean(),
    "100/genre (separate)": res_small.groupby("method")[["same_emotion@10", "mean_score@10"]].mean(),
    "300 pooled": res_pooled.groupby("method")[["same_genre@10", "same_emotion@10", "mean_score@10"]].mean(),
}).round(3)
compare

same_genre@10  same_emotion@10  mean_score@10
                     method                                                 
entire corpus        one-hot           0.622            0.333          0.317
                     tf-idf            0.533            0.289          0.278
                     word2vec          0.644            0.311          0.935
100/genre (separate) one-hot             NaN            0.378          0.042
                     tf-idf              NaN            0.378          0.034
                     word2vec            NaN            0.411          0.964
300 pooled           one-hot           0.411            0.278          0.084
                     tf-idf            0.411            0.278          0.069
                     word2vec          0.411            0.333          0.997